# From Literature to Benchmarks

## 1. Setup & imports
**Goal:** load the libraries, fix the random seeds, and set where files are read from and written to.

In [2]:
# Import all libraries used in the pipeline.
import os, re, json, html, random, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import torch
import pysbd
from bs4 import BeautifulSoup
from pylatexenc.latex2text import LatexNodes2Text
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
import textwrap

/home/s28rraoo_hpc/venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Fix the random seeds so the results are reproducible.
SEED = 42

In [4]:
# Input data and the folder where this notebook saves its results.
# Compressed INSPIRE release: one meta_<decade>.jsonl.zst file per decade (recid, title, abstract, categories, ...).
# Corpus folder: the downloaded corpus in ../../corpus (see corpus/README.md), or set LIT2BENCH_CORPUS.
SRC = os.environ.get("LIT2BENCH_CORPUS", "../../corpus")
OUTPUT_DIR = "baseline_outputs_notebook_03"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sentence-embedding model.
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

#sets the maximum number of characters that pandas will display for a single column's text
pd.set_option("display.max_colwidth", 120)


def out_path(name):
    """Full path of a file inside OUTPUT_DIR."""
    return os.path.join(OUTPUT_DIR, name)

## 2. Select a field and retrieve its related papers
**Goal:** find every paper related to the task

here we choose heavy-ion task so e.g. we choose the title or abstract names a heavy-ion collision system (Pb–Pb, p–Pb, Au–Au, Xe–Xe, O–O, ...) or a heavy-ion term (heavy-ion, quark–gluon plasma, ultra-peripheral, ...).

In [ ]:
# The compressed metadata files, one per decade. comment a line to skip that decade.(reading all decades takes may take up to 20 minutes)
FILES = [
    # "pre1990/meta_pre1990.jsonl.zst",
    # "1990s/meta_1990s.jsonl.zst",
    # "2000s/meta_2000s.jsonl.zst",
    "2010s/meta_2010s.jsonl.zst",
    "2020s/meta_2020s.jsonl.zst",
]


# All categories: Accelerators, Astrophysics, Computing, Condensed Matter, Data Analysis and Statistics,
# Experiment-HEP, Experiment-Nucl, General Physics, Gravitation and Cosmology, Instrumentation, Lattice,
# Math and Math Physics, Other, Phenomenology-HEP, Quantum Physics, Theory-HEP, Theory-Nucl
CATEGORIES = {"Experiment-Nucl", "Theory-Nucl", "Experiment-HEP", "Phenomenology-HEP"}

PBPB_FILE = out_path("pbpb_papers.jsonl")

In [ ]:
# Read one compressed .jsonl.zst file line by line
# and keep only the fields needed from each paper.
import subprocess

def read_meta(path):
    proc = subprocess.Popen(["zstd", "-dc", path], stdout=subprocess.PIPE, text=True, encoding="utf-8")
    try:
        for line in proc.stdout:
            if not line.strip():
                continue
            r = json.loads(line)
            yield {
                "recid": r.get("recid") or r.get("control_number"),
                "title": (r.get("titles") or [{}])[0].get("title"),
                "abstract": (r.get("abstracts") or [{}])[0].get("value"),
                "inspire_categories": sorted({c["term"] for c in r.get("inspire_categories") or [] if c.get("term")}),
                "year": (r.get("earliest_date") or "")[:4],
            }
    finally:
        proc.kill()
        proc.wait()

In [36]:
# Simplify LaTeX and Unicode spellings so "Pb--Pb", "$\mathrm{Pb}$-Pb", "Pb\!+\!Pb" and "Pb–Pb" all look the same to the search.
def normalize(t):
    t = re.sub(r"\\(?:mathrm|text|rm|mbox|textrm|mathbf)\s*", "", t)
    t = re.sub(r"\\(?:textendash|textemdash|times)", "-", t)
    t = re.sub(r"\\[!,;: ]", "", t)  # LaTeX spacing commands \! \, \; \: "\ "
    t = t.replace("$", "").replace("{", "").replace("}", "").replace("~", " ")
    t = re.sub(r"[\u2010-\u2015\u2212+/]|--", "-", t)  # dashes, minus, +, / -> '-'
    return t

In [37]:
# Search patterns for heavy-ion papers.
# ISO / ISO2 allow an optional mass number written before the symbol (^208Pb) or after it (Pb-208).
ISO = r"(?:\^?\s*\d{1,3}\s*)?"
ISO2 = r"(?:\s*-?\s*\d{1,3})?"

# Nuclei used as heavy-ion beams/targets, and light projectiles collided with them (p-Pb, d-Au, ...).
HEAVY = r"(?:Pb|Au|Xe|U|Cu|Zr|Ru|O|Ne|Ar|Kr|Sm|Ni|Ag|Al|Si|Ca|In)"
LIGHT = r"(?:p|d|He|" + HEAVY[3:]   # p, d, He or any heavy nucleus

PBPB = [
    # collision systems with a separator: Pb-Pb, Pb+Pb, Pb on Pb, p-Pb, d-Au, 129Xe-129Xe, ... 
    re.compile(rf"(?<![A-Za-z]){ISO}{LIGHT}{ISO2}\s*(?:-+|on)\s*{ISO}{HEAVY}{ISO2}(?![a-z])(?!\s*isotopes?)"),
    # collision systems written together: PbPb, pPb, Pbp, AuAu, dAu, XeXe, ...
    re.compile(r"\b(?:PbPb|pPb|Pbp|AuAu|pAu|dAu|XeXe|CuCu|CuAu|ZrZr|RuRu|UU|OO|NeNe|pO)\b"),
    # written out in words: lead-lead, proton-lead, gold-gold, collisions of lead nuclei, ...
    re.compile(r"\b(?:lead|gold|xenon|uranium|copper|proton|deuteron)\s*-\s*(?:lead|gold|xenon|uranium|copper)\b", re.I),
    re.compile(r"\b(?:collisions?|beams?)\s+of\s+(?:lead|gold|xenon|uranium|copper|oxygen|heavy)\s+(?:ions|nuclei)\b", re.I),
    # general heavy-ion terms
    re.compile(r"\bheavy\s*-?\s*ions?\b", re.I),
    re.compile(r"\bnucleus\s*-\s*nucleus\b|\bproton\s*-\s*nucleus\b", re.I),
    re.compile(r"\bquark\s*-?\s*gluon\s+plasma\b|\bQGP\b", re.I),
    re.compile(r"\bultra\s*-?\s*peripheral\b|\bUPCs?\b", re.I),
    re.compile(r"\b(?:A-A|p-A|AA|pA)\s+collisions?\b"),
]


def pbpb_hits(rec):
    """All heavy-ion mentions found in a paper's title + abstract."""
    text = normalize((rec.get("title") or "") + " \n " + (rec.get("abstract") or ""))
    return sorted({m.group(0) for rx in PBPB for m in rx.finditer(text)})

In [38]:
# Scan the decade files, keep papers in CATEGORIES, skip papers already seen,
# and write each heavy-ion paper to PBPB_FILE. 
seen, kept, no_abstract, per_category = set(), 0, 0, Counter()

with open(PBPB_FILE, "w", encoding="utf-8") as out:
    for fn in FILES:
        print("scanning", fn)
        for rec in read_meta(os.path.join(SRC, fn)):
            if CATEGORIES and not CATEGORIES & set(rec["inspire_categories"]):
                continue
            if rec["recid"] in seen:
                continue
            seen.add(rec["recid"])
            hits = pbpb_hits(rec)
            if not hits:
                continue
            out_rec = {
                "recid": rec["recid"],
                "title": rec["title"],
                "abstract": rec["abstract"],
                "inspire_categories": rec["inspire_categories"],
                "year": rec["year"],
                "matched_terms": hits,
            }
            out.write(json.dumps(out_rec, ensure_ascii=False) + "\n")
            kept += 1
            no_abstract += rec["abstract"] is None
            per_category.update(rec["inspire_categories"])

print(f"unique papers scanned: {len(seen)} | heavy-ion papers kept: {kept} | without abstract: {no_abstract}")

scanning 2010s/meta_2010s.jsonl.zst
scanning 2020s/meta_2020s.jsonl.zst
unique papers scanned: 268522 | heavy-ion papers kept: 26310 | without abstract: 1110


## 3. Keep only AI/ML papers
**Goal:** keep the Pb–Pb papers whose title or abstract mentions an AI/ML method, using a fixed list of phrases and acronyms.

In [39]:
# Load the Pb-Pb papers and drop those without an abstract.
df = pd.read_json(PBPB_FILE, lines=True)
df = df[df["abstract"].notna() & (df["abstract"].str.strip() != "")]
df = df.drop_duplicates(subset="recid").reset_index(drop=True)
print("Pb-Pb papers with abstract:", len(df))

Pb-Pb papers with abstract: 25200


In [40]:
# Phrases and acronyms that signal AI/ML. (you can add or delete items from these lists to suit your needs.)
AI_PHRASES = [
    "machine learning", "deep learning", "neural network", "neural networks",
    "neural net", "convolutional", "recurrent neural", "graph neural",
    "artificial intelligence", "boosted decision tree", "boosted decision trees",
    "random forest", "gradient boosting", "xgboost", "autoencoder", "autoencoders",
    "generative adversarial", "normalizing flow", "normalizing flows", "normalising flow",
    "foundation model", "foundation models", "self-supervised",
    "unsupervised learning", "supervised learning", "reinforcement learning",
    "transfer learning", "attention mechanism", "transformer network",
    "transformer model", "language model", "large language model",
    "support vector machine", "multivariate analysis", "multivariate classifier",
    "particle transformer", "lstm", "pytorch", "tensorflow", 
    "k-means", "DBSCAN", "Gaussian mixture", "unsupervised clustering",
    "contrastive learning", "domain adaptation",
    "feature engineering", "hyperparameter tuning", 
    "few-shot learning", "zero-shot learning", "meta-learning", "ensemble learning",
    "deep neural network", "deep neural networks",
    "multilayer perceptron", "multilayer perceptrons", "residual network", "ResNet",
    "vision transformer", "vision transformers", "mixture of experts", "state space model", "encoder-decoder",
    "variational autoencoder", "variational autoencoders", "graph convolutional network", "graph attention network",
    "geometric deep learning", "graph neural network", "graph neural networks", "generative model", "generative models",
    "instruction tuning", "chain of thought", "embedding model", 
    "vector embedding", "Gaussian process", "Bayesian optimization", "decision tree", "decision trees", "AdaBoost",
    "anomaly detection", "physics-informed neural", "Kolmogorov-Arnold", "symbolic regression",
]

AI_ACRONYMS = ["AI", "CNN", "GNN", "LSTM", "BDT", "BDTs", "DNN", "ANN", "GAN",
               "GANs", "VAE", "VAEs", "RNN", "MLP", "SVM", "LLM", "LLMs", "SFT",
               "RAG", "ViT", "ViTs", "LoRA", "RLHF", "GCN", "GCNs", "DQN", "UNet"]

In [41]:
# search pattern per list
def _flexible(p):
    return re.escape(p).replace(r"\ ", r"[\s-]+").replace(r"\-", r"[\s-]*")

phrase_re = re.compile(r"\b(?:" + "|".join(_flexible(p) for p in AI_PHRASES) + r")\b", flags=re.IGNORECASE)
acronym_re = re.compile(r"\b(?:" + "|".join(AI_ACRONYMS) + r")\b")


def find_ai_terms(text):
    """Sorted list of AI/ML terms found in a text."""
    found = {re.sub(r"[\s-]+", " ", m.lower()) for m in phrase_re.findall(text)}
    found |= set(acronym_re.findall(text))
    return sorted(found)

In [42]:
# Tag every paper with the AI/ML terms in its title + abstract, and show the most frequent terms.
text = df["title"].fillna("") + ". " + df["abstract"]
df["ai_terms"] = text.map(find_ai_terms)
print("papers mentioning at least one AI/ML term:", (df["ai_terms"].map(len) > 0).sum(), "of", len(df))
df["ai_terms"].explode().value_counts().head(25)

papers mentioning at least one AI/ML term: 360 of 25200


ai_terms
machine learning           174
neural network              78
deep learning               69
neural networks             46
convolutional               37
gaussian process            30
deep neural network         27
artificial intelligence     16
CNN                         15
AI                          13
DNN                         12
deep neural networks        11
boosted decision trees      10
ANN                         10
gradient boosting           10
BDT                          9
supervised learning          9
multivariate analysis        8
transfer learning            7
physics informed neural      6
xgboost                      6
decision tree                6
boosted decision tree        6
graph neural                 6
generative models            5
Name: count, dtype: int64

In [43]:
# Keep only papers that mention at least one AI/ML term, then save them for the next steps.
ai_df = df[df["ai_terms"].map(len) > 0].reset_index(drop=True)
ai_df.to_json(out_path("papers_ai_only.jsonl"), orient="records", lines=True, force_ascii=False)
print(f"AI/ML papers kept: {len(ai_df)} of {len(df)} ({len(ai_df) / len(df):.1%})")

AI/ML papers kept: 360 of 25200 (1.4%)


## 4. Clean the text
**Goal:** turn LaTeX formulas and HTML markup (e.g. `$\sqrt{s_{NN}}$`, `<sub>`) into plain text, so the language model reads words

In [44]:
# Tools for the cleaning: a LaTeX-to-text converter and patterns for $...$ math and HTML <sub>/<sup> tags.
_l2t = LatexNodes2Text(math_mode="text")
_MATH_RE = re.compile(r"\$\$?(.+?)\$\$?", re.S)
_SUBSUP_RE = re.compile(r"<(sub|sup)>(.*?)</\1>", re.I | re.S)


def _safe_l2t(s):
    """Convert a LaTeX snippet to text; if the converter fails, just strip the LaTeX commands."""
    try:
        return _l2t.latex_to_text(s)
    except Exception:
        return re.sub(r"\\[A-Za-z]+\*?", "", s).replace("{", "").replace("}", "")

In [45]:
# Clean one title or abstract: HTML -> text, fix special characters, LaTeX -> text, collapse spaces.
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # 1. HTML: <sub>x</sub> -> _x, <sup>x</sup> -> ^x; <math> blocks -> their TeX version; drop other tags
    text = _SUBSUP_RE.sub(lambda m: ("_" if m[1].lower() == "sub" else "^") + m[2], text)
    soup = BeautifulSoup(text, "html.parser")
    for math in soup.find_all("math"):
        ann = math.find("annotation", attrs={"encoding": re.compile("tex", re.I)})
        math.replace_with(ann.get_text(strip=True) if ann else math.get_text(""))
    text = soup.get_text(" ", strip=True)
    # 2. HTML entities (&lt; -> <) and Unicode normalisation
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    # 3. LaTeX -> text: first the $...$ math parts, then the rest
    text = _MATH_RE.sub(lambda m: _safe_l2t(m[1]), text)
    try:
        text = _l2t.latex_to_text(text)
    except Exception:  # the converter crashes on some broken LaTeX (e.g. \href); strip it crudely instead
        text = re.sub(r"\\href\s*\{[^{}]*\}", "", text)
        text = re.sub(r"\\[A-Za-z]+\*?", "", text).replace("{", "").replace("}", "").replace("$", "")
    # 4. whitespace
    return re.sub(r"\s+", " ", text).strip()

In [46]:
# Clean all titles and abstracts (the original text stays in the "title"/"abstract" columns).
ai_df["clean_title"] = ai_df["title"].fillna("").map(clean_text)
ai_df["clean_abstract"] = ai_df["abstract"].fillna("").map(clean_text)
ai_df["clean_document"] = (ai_df["clean_title"] + ". " + ai_df["clean_abstract"]).str.strip()
ai_df.to_json(out_path("papers_ai_cleaned.jsonl"), orient="records", lines=True, force_ascii=False)
print("papers cleaned:", len(ai_df))

papers cleaned: 360


## 5. Turn papers into vectors (embeddings)
**Goal:** represent each paper as a list of 384 numbers (an *embedding*) so that papers about similar things get similar vectors.

The model (MiniLM) reads at most ~256 tokens (word pieces) at once, and many abstracts are longer. So we cut each paper into
overlapping *windows* of 250 tokens (50 tokens overlap), embed every window, and average the windows of a paper.

In [47]:
# Drop papers whose cleaned abstract is empty, then build one text per paper: "title. abstract".
papers = ai_df[ai_df["clean_abstract"].str.strip() != ""].drop_duplicates(subset="recid").reset_index(drop=True)
docs = (papers["clean_title"].fillna("") + ". " + papers["clean_abstract"]).tolist()
print("papers to embed:", len(docs))

papers to embed: 360


In [4]:
# Load the sentence-embedding model (downloaded on first use, then cached).
encoder = SentenceTransformer(EMBED_MODEL, device="cpu")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5837.15it/s]


In [13]:
# Split a text into overlapping windows of at most MAX_TOKENS tokens; short texts stay as one window.
MAX_TOKENS = 250
OVERLAP = 50


def make_windows(text, tokenizer, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    offsets = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    n = len(offsets)
    if n <= max_tokens:
        return [text]
    windows, start, step = [], 0, max_tokens - overlap
    while start < n:
        end = min(start + max_tokens, n)
        windows.append(text[offsets[start][0]:offsets[end - 1][1]])
        if end == n:
            break
        start += step
    return windows

In [50]:
# Cut every paper into windows and remember which paper each window belongs to.
window_texts, window_owner = [], []
for i, doc in enumerate(docs):
    for w in make_windows(doc, encoder.tokenizer):
        window_texts.append(w)
        window_owner.append(i)
window_owner = np.array(window_owner)
print(len(docs), "papers ->", len(window_texts), "windows")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (285 > 256). Running this sequence through the model will result in indexing errors


360 papers -> 599 windows


In [51]:
# Embed all windows; normalize_embeddings=True makes every vector length 1.
window_emb = encoder.encode(window_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

Batches: 100%|██████████| 10/10 [00:02<00:00,  4.51it/s]


In [52]:
# Average the windows of each paper into one vector, normalize it to length 1 again, and save it.
embeddings = np.zeros((len(docs), window_emb.shape[1]), dtype=np.float32)
np.add.at(embeddings, window_owner, window_emb)
embeddings = embeddings / np.bincount(window_owner, minlength=len(docs)).reshape(-1, 1) 
embeddings = embeddings / np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
np.save(out_path("paper_embeddings.npy"), embeddings)
print("embedding matrix:", embeddings.shape)

embedding matrix: (360, 384)


## 6. Find topics with BERTopic
**Goal:** group the papers into topics automatically and describe each topic with its most typical words.

BERTopic does three things:
1. **UMAP** squeezes the 384-number vectors down to 5 numbers (clustering works badly in very many dimensions).
2. **HDBSCAN** finds dense groups of papers in that 5-D space. Papers that fit no group get topic **-1** ("outliers").
3. **c-TF-IDF** finds the words that are frequent in one topic but rare in the others, to name the topic.

In [53]:
# Smallest number of papers that may form a topic.
MIN_TOPIC_SIZE = 4
TOPIC_DIR = out_path("bertopic")
os.makedirs(TOPIC_DIR, exist_ok=True)

In [54]:
# Step 1: UMAP to 5 dimensions. "cosine" compares the direction of text vectors
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)

In [55]:
# Step 2: HDBSCAN clustering. min_samples=2 and the "leaf" method favour many small, fine-grained topics.
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, metric="euclidean", min_samples=2,
                        cluster_selection_method="leaf", prediction_data=True)

In [56]:
# Step 3: word counts for topic names; uses single words and word pairs, without English filler words.
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1,
                                   token_pattern=r"(?u)\b\w[\w-]*\b")

In [57]:
# Put the three parts together and fit them on our papers and their embeddings.
topic_model = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer_model, verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

2026-11-12 12:12:09,468 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 12:12:17,555 - BERTopic - Dimensionality - Completed ✓
2026-11-12 12:12:17,557 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 12:12:17,564 - BERTopic - Cluster - Completed ✓
2026-11-12 12:12:17,566 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 12:12:17,681 - BERTopic - Representation - Completed ✓


In [58]:
# Attach the topic of each paper and save the overview tables and the model.
topic_info = topic_model.get_topic_info()
topic_names = dict(zip(topic_info["Topic"], topic_info["Name"]))
papers["topic"] = topics
papers["topic_name"] = papers["topic"].map(topic_names)

topic_info.to_csv(os.path.join(TOPIC_DIR, "topics_overview.csv"), index=False)
papers[["recid", "title", "topic", "topic_name"]].to_csv(os.path.join(TOPIC_DIR, "papers_with_topics.csv"), index=False)
topic_model.save(os.path.join(TOPIC_DIR, "bertopic_model"))

print("topics found:", (topic_info["Topic"] != -1).sum(), "| outlier papers (topic -1):", (papers["topic"] == -1).sum())
topic_info[["Topic", "Count", "Name"]]

2026-11-12 12:12:21,331 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


topics found: 37 | outlier papers (topic -1): 95


,Topic,Count,Name
0,-1,95,-1_collisions_data_using_heavy
1,0,16,0_fusion_cross_fusion cross_nuclear
2,1,13,1_collisions_production_mesons_pp
3,2,12,2_mass_dielectron_muon_invariant
4,3,11,3_network_neural_relativistic_hydrodynamics
5,4,11,4_ml_impact_impact parameter_parameter
6,5,10,5_cnn_impact_impact parameter_models
7,6,10,6_particle identification_identification_particle_bmn
8,7,10,7_jet_jets_background_momenta
9,8,9,8_star_auau collisions_auau_d


In [59]:
# Print the top words and example titles of every topic.
def show_topic(topic_id, n_titles=8):
    print("=" * 80)
    print("topic", topic_id, "|", topic_names.get(topic_id), "|", (papers["topic"] == topic_id).sum(), "papers")
    print("top words:", [w for w, _ in topic_model.get_topic(topic_id)][:10])
    for title in papers.loc[papers["topic"] == topic_id, "clean_title"].head(n_titles):
        print("   -", title[:100])


for t in topic_info.loc[topic_info["Topic"] != -1, "Topic"].head(15):
    show_topic(t)

topic 0 | 0_fusion_cross_fusion cross_nuclear | 16 papers
top words: ['fusion', 'cross', 'fusion cross', 'nuclear', 'barrier', 'cross section', 'section', 'data', 'artificial', 'intelligence']
   - Neutron separation energies from nuclear mass systematics using neural networks
   - Advancing Nuclear Physics with Machine Learning and Artificial Intelligence
   - Bayesian Inference for Extracting Barrier Distributions from Fusion Excitation Functions
   - Studying the incomplete fusion with physics-constrained machine learning approaches
   - Predicting the formation probability PCN of a compound nucleus with machine learning
   - A fast Bayesian surrogate for the photon flux in ultra-peripheral collisions
   - AI-assisted transport of radioactive ion beams
   - Toward a better characterization of the nuclear EOS using central collisions around Fermi energy
topic 1 | 1_collisions_production_mesons_pp | 13 papers
top words: ['collisions', 'production', 'mesons', 'pp', 'ds', 'd0', 'measure

## 7. Pick one topic
**Goal:** choose the topic whose papers you want to analyse in detail in the rest of the notebook.

### Your turn: choose a topic
**Look at:** the topic table and the top words / example titles.
**Decide:** which topic to follow. For the question "which tasks does ML solve?"

Set `selected_topic` to the topic number you want to investigate more. (you can choose one or more clusters that match your preferred topic)
Choose by reading the words and titles.

In [60]:
selected_topics = [7, 13, 19, 31, 33, 34]   # e.g.topic relate to jets  # <-- SET THIS to a topic number from the table above

In [62]:
# Keep only the papers of the chosen topic and save them.
assert selected_topics is not None, "Set selected_topic in the 'Your turn' cell above first."
assert selected_topics != -1 and all(topic in topic_names for topic in selected_topics), f"One or more selected topics are not real topics in the table."

topic_papers = papers[papers["topic"].isin(selected_topics)].reset_index(drop=True)
topic_papers.to_json(out_path(f"topic_{'_'.join(map(str, selected_topics))}_papers.jsonl"), orient="records", lines=True, force_ascii=False)

print(f"topic {selected_topics} | {', '.join(topic_names[topic] for topic in selected_topics)} | {len(topic_papers)} papers")
for title in topic_papers["clean_title"]:
    print("   -", title[:100])

topic [7, 13, 19, 31, 33, 34] | 7_jet_jets_background_momenta, 13_jet_background_subtraction_jets, 19_jets_jet_quenching_combinatorial jets, 31_mach_jet production_jetinduced_jet, 33_jet_quenched jets_quenched_substructure, 34_tagging_bjet_bjet tagging_graph | 36 papers
   - Machine Learning based jet momentum reconstruction in Pb-Pb collisions measured with the ALICE detec
   - Looking inside jets: an introduction to jet substructure and boosted-object phenomenology
   - Machine Learning based jet momentum reconstruction in heavy-ion collisions
   - Study of jet-induced hydro response in high-energy heavy-ion collisions with a flow-matching generat
   - Jet quenching identification via supervised learning in simulated heavy-ion collisions
   - B-hadron identification in b-jets using novel deep learning technique in pp and PbPb collisions in C
   - Quantifying vacuum-like jets in heavy-ion collisions: a machine learning study
   - Validating a machine learning approach to identify quen

## 8. Extract task sentences
**Goal:** for each paper, find the abstract sentences that say *what the paper does* (e.g. "we estimate …", "is used to …").

Each paper is then described by its **title** plus up to 3 such **task sentences** (its *task context*).

In [ ]:
# Sentence splitter for scientific English.
seg = pysbd.Segmenter(language="en", clean=False)

In [8]:
# Phrases that usually introduce the goal of a paper ("we propose", "is used to", "we estimate", ...).
OBJECTIVE_PATTERNS = [
    r"\bwe (propose|develop|introduce|present|investigate|study)\b",
    r"\bwe (aim|seek|attempt|evaluate|demonstrate)\b",
    r"\bour (goal|objective|aim|approach|method)\b",
    r"\b(in order to|designed to|used to|aimed at)\b",
    r"\bwe (show|apply|train|use|perform)\b",
    r"\b(is|are|was|were) (proposed|developed|introduced|presented|investigated|studied|applied|utilized|employed|demonstrated|evaluated)\b",
    r"\b(paper|study|work|approach|framework|method) (proposes|develops|introduces|presents|investigates|studies|aims|demonstrates)\b",
    r"\bwe (estimate|predict|extract|reconstruct|map)\b",
    r"\b(is|are) used to\b",
    r"\b(estimate|predict|extract)\b",
]

In [9]:
# Abstract sentences (6+ words) that match a pattern; if none match, fall back to the first 6+ word sentence.
def objective_sentences(abstract):
    abstract = abstract if isinstance(abstract, str) else ""
    sentences = [s.strip() for s in seg.segment(abstract)] if abstract.strip() else []
    long_sentences = [s for s in sentences if len(s.split()) >= 6]
    hits = [s for s in long_sentences if any(re.search(p, s, re.I) for p in OBJECTIVE_PATTERNS)]
    return hits if hits else long_sentences[:1]

In [10]:
# One row per paper: its title and its task context.
tasks = pd.DataFrame({
    "paper_index": topic_papers.index,
    "recid": topic_papers["recid"],
    "title": topic_papers["clean_title"].fillna(""),
    "task_context": topic_papers["clean_abstract"].map(lambda a: " ".join(objective_sentences(a)[:3])),
})
tasks[["title", "task_context"]].head()

,title,task_context
0,Machine Learning based jet momentum reconstruction in Pb-Pb collisions measured with the ALICE detector,The approach allows the measurement of jets down to extremely low transverse momenta and for large resolution R by m...
1,Looking inside jets: an introduction to jet substructure and boosted-object phenomenology,"Furthermore, while first-principle calculations in QCD have successfully been applied in order to understand and cha..."
2,Machine Learning based jet momentum reconstruction in heavy-ion collisions,"In this paper, a novel approach to correct jet momenta (or energies) for the underlying background in heavy-ion coll..."
3,Study of jet-induced hydro response in high-energy heavy-ion collisions with a flow-matching generative model,"In this study, we develop a flow matching generative model trained by CoLBT-hydro events for the study of the medium..."
4,Jet quenching identification via supervised learning in simulated heavy-ion collisions,"In this work, we apply sequential machine-learning architectures to the jet declustering history tree, achieving imp..."


## 9. Embed the task descriptions
**Goal:** turn each paper's title and task context into one vector, so we can compare papers by the task they solve.

In [14]:
# Embed titles and task contexts separately.
title_emb = encoder.encode(tasks["title"].tolist(), batch_size=32, normalize_embeddings=True)

# Embed texts that may be longer than the model's limit: split into windows, embed, and average per text.
def embed_windowed(texts):
    owner, windows = [], []
    for i, text in enumerate(texts):
        for w in make_windows(text, encoder.tokenizer):
            windows.append(w)
            owner.append(i)
    owner = np.array(owner)
    emb = encoder.encode(windows, batch_size=32, normalize_embeddings=True)
    out = np.zeros((len(texts), emb.shape[1]))
    np.add.at(out, owner, emb)
    out = out / np.bincount(owner, minlength=len(texts)).reshape(-1, 1)
    return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-12)


context_emb = embed_windowed(tasks["task_context"].tolist())

In [15]:
# Mix them 60% title + 40% context (the title states the task most directly); use the title alone if there is no context.
TITLE_WEIGHT, CONTEXT_WEIGHT = 0.60, 0.40
task_embeddings = TITLE_WEIGHT * title_emb + CONTEXT_WEIGHT * context_emb

no_context = (tasks["task_context"].str.strip() == "").to_numpy()
task_embeddings[no_context] = title_emb[no_context]
task_embeddings = task_embeddings / np.maximum(np.linalg.norm(task_embeddings, axis=1, keepdims=True), 1e-12)
print("task embeddings:", task_embeddings.shape, "| papers without task context:", no_context.sum())

task embeddings: (36, 384) | papers without task context: 0


## 10. Cluster papers by task
**Goal:** group papers that solve similar tasks, using *agglomerative clustering*.

Agglomerative clustering starts with every paper as its own group and keeps merging the two closest groups. The **distance
threshold** says when to stop: groups further apart than the threshold are not merged. Distance here is *cosine distance*
(0 = same meaning, 1 = unrelated). A small threshold gives many small, tight clusters; a large one gives few broad clusters.

In [16]:
# Cluster the task embeddings for a given distance threshold; returns one cluster label per paper.
def cluster_tasks(emb, threshold):
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=threshold, metric="cosine", linkage="average")
    return model.fit_predict(emb)

In [17]:
# Try several thresholds and compare the number and sizes of the clusters.
rows = []
for th in [0.30, 0.40, 0.50, 0.60, 0.70]:
    sizes = np.bincount(cluster_tasks(task_embeddings, th))
    rows.append({"threshold": th, "n_clusters": len(sizes), "largest": sizes.max(), "single_paper_clusters": (sizes == 1).sum()})
pd.DataFrame(rows)

,threshold,n_clusters,largest,single_paper_clusters
0,0.3,13,8,5
1,0.4,6,27,3
2,0.5,2,35,1
3,0.6,1,36,0
4,0.7,1,36,0


In [18]:
# Look at which titles end up together for one threshold. Change preview_threshold and re-run to compare.
preview_threshold = 0.30
preview = tasks.assign(cluster=cluster_tasks(task_embeddings, preview_threshold))
for cid, group in preview.groupby("cluster"):
    print(f"cluster {cid} ({len(group)} papers)")
    for title in group["title"]:
        print("   -", title[:100])

cluster 0 (2 papers)
   - Measurement of the radius dependence of charged-particle jet suppression in Pb–Pb collisions at sNN=
   - Measurements of Nuclear Modification Factor of Inclusive Full Jet Measurements in Pb-Pb Collisions a
cluster 1 (8 papers)
   - Machine Learning based jet momentum reconstruction in heavy-ion collisions
   - Jet quenching identification via supervised learning in simulated heavy-ion collisions
   - Quantifying vacuum-like jets in heavy-ion collisions: a machine learning study
   - Validating a machine learning approach to identify quenched jets in heavy-ion collisions
   - Model-Agnostic Tagging of Quenched Jets in Heavy-Ion Collisions
   - Identifying quenched jets in heavy ion collisions with machine learning
   - Jet characterization in Heavy Ion Collisions by QCD-Aware Graph Neural Networks
   - Deep learning jet modifications in heavy-ion collisions
cluster 2 (3 papers)
   - Deep image reconstruction for background subtraction in heavy-ion collisions
 

### Your turn: choose the distance threshold
**Look at:** the threshold table and the grouped titles above.
**Decide:** which threshold gives clusters where the papers really share a task

Set `distance_threshold` to a number between 0 and 1.

In [ ]:
distance_threshold = 0.3  # <-- SET THIS 

In [20]:
# Cluster the papers with the chosen threshold.
assert distance_threshold is not None, "Set distance_threshold in the 'Your turn' cell above first."
assert 0 < distance_threshold < 1, "distance_threshold should be between 0 and 1."

tasks["cluster"] = cluster_tasks(task_embeddings, distance_threshold)
print(f"{len(tasks)} papers -> {tasks['cluster'].nunique()} task clusters (threshold {distance_threshold})")

36 papers -> 13 task clusters (threshold 0.3)


## 11. Describe the task clusters
**Goal:** give every cluster a representative title and a few keywords that describe its task.

In [21]:
# Representative title = the paper most similar, on average, to the other papers in its cluster.
summary_rows = []
for cid, group in tasks.groupby("cluster"):
    emb = task_embeddings[group.index]
    best = group.index[np.argmax(cosine_similarity(emb).mean(axis=1))]
    summary_rows.append({"cluster": cid, "n_papers": len(group), "representative_title": tasks.loc[best, "title"]})
summary = pd.DataFrame(summary_rows)
summary.sort_values("n_papers", ascending=False)

,cluster,n_papers,representative_title
1,1,8,Validating a machine learning approach to identify quenched jets in heavy-ion collisions
3,3,7,Overview: Jet quenching with machine learning
2,2,3,Multiplicity Based Background Subtraction for Jets in Heavy Ion Collisions
4,4,3,Machine learning-based b-jet tagging in pp collisions at s=13 TeV
6,6,3,Jet tomography in hot QCD medium with deep learning
10,10,3,Machine Learning based jet momentum reconstruction in Pb-Pb collisions measured with the ALICE detector
0,0,2,Measurements of Nuclear Modification Factor of Inclusive Full Jet Measurements in Pb-Pb Collisions at √(s_NN) = 5.02...
5,5,2,Study of jet-induced hydro response in high-energy heavy-ion collisions with a flow-matching generative model
7,7,1,Looking inside jets: an introduction to jet substructure and boosted-object phenomenology
8,8,1,Underlying Event Fluctuations and Jet Background in √sNN = 200 GeV Au+Au Collisions Using the sPHENIX Detector


To find keywords we need to ignore words that every heavy-ion paper uses (they say nothing about the task) and the names
of ML methods (they say *how*, not *what*). The lists below were chosen by hand for this field; for other choosen fileds it should be modified.

In [22]:
# Words used in nearly every Pb-Pb / heavy-ion paper -> they say nothing about the task.
DOMAIN_STOP = {
    # collision system / facilities / experiments
    "heavy", "ion", "ions", "heavy-ion", "heavy-ions", "pb-pb", "pbpb", "pb", "lead", "xe-xe", "au-au",
    "nucleus", "nuclei", "nuclear", "relativistic", "ultrarelativistic", "ultra-relativistic",
    "collision", "collisions", "collider", "lhc", "rhic", "sps", "cern", "bnl", "alice", "cms", "atlas",
    "lhcb", "star", "phenix", "experiment", "experiments", "experimental", "detector", "measured",
    "measurement", "measurements", "recorded", "collected", "run",
    # units / kinematics
    "tev", "gev", "energy", "energies", "nn", "sqrt", "s_nn", "center-of-mass", "pp", "p-pb",
    # generic physics words
    "physics", "particle", "particles", "matter", "medium", "system", "systems", "events", "event",
    "simulation", "simulations", "simulated", "samples", "sample", "observable", "observables",
    # writing filler
    "present", "presented", "propose", "proposed", "study", "studies", "studied", "paper", "work",
    "approach", "method", "methods", "results", "result", "using", "based", "new", "novel",
    "introduce", "develop", "developed", "show", "shows", "demonstrate", "performance", "note",
    "use", "used", "recent", "high", "large", "significant", "significantly", "improve",
    "improvements", "effectiveness", "showcasing", "achieves", "advances", "traditional",
    "development", "precision", "highly", "considered", "various", "different", "order", "provide",
    "provides", "obtained", "good", "agreement", "framework", "first", "data", "input",
}

In [23]:
# Names of ML methods -> they say HOW, not WHAT the task is. 
METHOD_STOP = {
    "machine", "learning", "deep", "neural", "network", "networks", "artificial", "ai", "ml", "ann",
    "dnn", "cnn", "cnns", "gnn", "gan", "gans", "convolutional", "feed-forward", "feedforward",
    "recurrent", "graph", "transformer", "transformers", "attention", "decision", "trees", "tree",
    "bdt", "boosted", "xgboost", "random", "forest", "adversarial", "diffusion", "point", "cloud",
    "clouds", "layer", "layers", "architecture", "architectures", "trained", "training", "train",
    "model", "models", "modeling", "modelling", "algorithm", "algorithms", "technique", "techniques",
    "tool", "tools", "statistical", "data-driven", "deep-learning", "learning-based",
    "network-based", "ml-based", "accuracy", "efficient", "efficiency", "applied", "application",
    "applications", "implementation",
}

In [24]:
# Full stop-word list (English filler + the two lists; "top"/"bottom" are kept because they can be quark names)
# and the token rule: words of 2+ letters or letter and digits, hyphens allowed.
STOP = list((ENGLISH_STOP_WORDS - {"top", "bottom"}) | DOMAIN_STOP | METHOD_STOP)
TOKEN = TOKEN = r"(?u)\b[a-zA-Z][a-zA-Z0-9\-]+\b"

In [25]:
# Candidate keywords = single words and word pairs from the titles, minus generic action words ("analysis", "compare", ...).
GENERIC_STEMS = ("analy", "evaluat", "introduc", "investigat", "compar", "present", "focus", "appl",
                 "train", "enhanc", "simultaneous", "explor", "fully", "multiple", "standard")

title_vocab = CountVectorizer(stop_words=STOP, ngram_range=(1, 2), min_df=1, max_df=1.0,
                              token_pattern=TOKEN).fit(tasks["title"]).vocabulary_
vocab = [t for t in sorted(title_vocab) if not any(w.startswith(GENERIC_STEMS) for w in t.split())]
print(len(vocab), "candidate keywords")

187 candidate keywords


In [26]:
# Score the keywords per cluster with TF-IDF: high score = frequent in this cluster's titles + task context, rare elsewhere.
cluster_docs = (tasks["title"] + " " + tasks["task_context"]).groupby(tasks["cluster"]).apply(" ".join)
tfidf = TfidfVectorizer(vocabulary=vocab, stop_words=STOP, ngram_range=(1, 2), sublinear_tf=True, token_pattern=TOKEN)
scores = tfidf.fit_transform(cluster_docs.values)
terms = np.array(tfidf.get_feature_names_out())

In [27]:
# Pick the top keywords of a cluster, skipping any keyword that repeats a word already picked.
TOP_K = 5


def pick_terms(row_scores, k=TOP_K):
    picked, used = [], set()
    for j in np.argsort(-row_scores):
        if row_scores[j] <= 0 or len(picked) == k:
            break
        words = {w.rstrip("s") for w in terms[j].split()}
        if words & used:
            continue
        picked.append(terms[j])
        used |= words
    return ", ".join(picked)

In [28]:
# Add keywords to the summary; single-paper clusters are marked "(too small)", clusters without keywords "(no task words)".
MIN_PAPERS = 2
keywords = {cid: pick_terms(scores[i].toarray().ravel()) for i, cid in enumerate(cluster_docs.index)}
summary["top_terms"] = summary["cluster"].map(keywords).fillna("")
summary.loc[summary["top_terms"].str.strip() == "", "top_terms"] = "(no task words)"
summary.loc[summary["n_papers"] < MIN_PAPERS, "top_terms"] = "(too small)"
summary.sort_values("n_papers", ascending=False)

,cluster,n_papers,representative_title,top_terms
1,1,8,Validating a machine learning approach to identify quenched jets in heavy-ion collisions,"momentum, quenched, jet, quenching, modifications"
3,3,7,Overview: Jet quenching with machine learning,"gluon plasma, quark, jet quenching, overview, qgp"
2,2,3,Multiplicity Based Background Subtraction for Jets in Heavy Ion Collisions,"background subtraction, multiplicity, interpretable, jet, image"
4,4,3,Machine learning-based b-jet tagging in pp collisions at s=13 TeV,"b-jet tagging, b-hadron, gnn-based, identification, contamination"
6,6,3,Jet tomography in hot QCD medium with deep learning,"jet tomography, assisted, hot, cones, mach"
10,10,3,Machine Learning based jet momentum reconstruction in Pb-Pb collisions measured with the ALICE detector,"momentum, qgp background, jet, reconstruction, inclusive"
0,0,2,Measurements of Nuclear Modification Factor of Inclusive Full Jet Measurements in Pb-Pb Collisions at √(s_NN) = 5.02...,"jet suppression, inclusive, snn, dependence charged-particle, factor"
5,5,2,Study of jet-induced hydro response in high-energy heavy-ion collisions with a flow-matching generative model,"high-energy, response, suppression, generative, qgp"
7,7,1,Looking inside jets: an introduction to jet substructure and boosted-object phenomenology,(too small)
8,8,1,Underlying Event Fluctuations and Jet Background in √sNN = 200 GeV Au+Au Collisions Using the sPHENIX Detector,(too small)


## 12. Save and review the results
**Goal:** save one table per paper and one per cluster, and list the clusters that describe a real, shared task.

In [29]:
# One row per paper (with its cluster) and one row per cluster (with keywords and all its titles).
tasks.to_csv(out_path("papers_with_task_clusters.csv"), index=False)

titles_per_cluster = tasks.groupby("cluster")["title"].apply(" | ".join).rename("papers")
clusters_table = summary.join(titles_per_cluster, on="cluster").sort_values("n_papers", ascending=False)
clusters_table.to_csv(out_path("task_clusters.csv"), index=False)
print("saved to", os.path.abspath(OUTPUT_DIR))

saved to /home/s28rraoo_hpc/hackaton/baseline_outputs_notebook_03


## 13. Choose a task cluster and write the findings in JSON file
**Goal:** pick the task cluster you want to read by hand to define  downstream task, and save its papers to one JSON file.

### Your turn: choose the task cluster
**Look at:** the cluster summary above (keywords + representative title) and the grouped titles.
**Decide:** which cluster describes one clear task.
You can give several cluster numbers, e.g. `[2, 3]`, if they are discribe a close topic.

In [30]:
selected_clusters = [1]  # <-- SET THIS to cluster number(s) from the summary table above

In [31]:
# Collect the papers of the chosen cluster(s) with everything needed to read them by hand, and write one JSON file.
assert all(c in set(tasks["cluster"]) for c in selected_clusters), "One or more selected clusters do not exist."

chosen = tasks[tasks["cluster"].isin(selected_clusters)]
chosen = chosen.drop_duplicates(subset="title")  # the same paper can appear as two INSPIRE records

final_papers = []
for _, row in chosen.iterrows():
    paper = topic_papers.loc[row["paper_index"]]
    final_papers.append({
        "recid": int(row["recid"]),
        "inspire_url": f"https://inspirehep.net/literature/{int(row['recid'])}",
        "title": row["title"],
        "year": str(paper.get("year", "")),
        "inspire_categories": list(paper["inspire_categories"]),
        "abstract": paper["clean_abstract"],
        "task_context": row["task_context"],
        "ai_terms": list(paper["ai_terms"]),
        "task_cluster": int(row["cluster"]),
    })

final = {
    "selection": {
        "bertopic_topics": [int(t) for t in selected_topics],
        "distance_threshold": distance_threshold,
        "task_clusters": [int(c) for c in selected_clusters],
        "cluster_keywords": {int(c): keywords.get(c, "") for c in selected_clusters},
        "n_papers": len(final_papers),
    },
    "papers": final_papers,
}

FINAL_FILE = out_path(f"final_task_cluster_{'_'.join(map(str, selected_clusters))}.json")
with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(final, f, indent=2, ensure_ascii=False)

print(f"saved {len(final_papers)} papers to {os.path.abspath(FINAL_FILE)}")
for p in final_papers:
    print(f"   - [{p['year']}] {p['title'][:90]}  ({p['inspire_url']})")

saved 8 papers to /home/s28rraoo_hpc/hackaton/baseline_outputs_notebook_03/final_task_cluster_1.json
   - [2018] Machine Learning based jet momentum reconstruction in heavy-ion collisions  (https://inspirehep.net/literature/1698396)
   - [2026] Jet quenching identification via supervised learning in simulated heavy-ion collisions  (https://inspirehep.net/literature/3148534)
   - [2025] Quantifying vacuum-like jets in heavy-ion collisions: a machine learning study  (https://inspirehep.net/literature/3083458)
   - [2025] Validating a machine learning approach to identify quenched jets in heavy-ion collisions  (https://inspirehep.net/literature/3080476)
   - [2024] Model-Agnostic Tagging of Quenched Jets in Heavy-Ion Collisions  (https://inspirehep.net/literature/2854166)
   - [2022] Identifying quenched jets in heavy ion collisions with machine learning  (https://inspirehep.net/literature/2091324)
   - [2021] Jet characterization in Heavy Ion Collisions by QCD-Aware Graph Neural Networks

## 14. Manual review: read each paper and fill in its benchmark fields
Start with the title and abstract. For each paper, record what the model takes as input and what it predicts for one object or event. Record the evaluation metric and numerical result separately. Check the full text when needed; keep an unresolved match as a candidate.

Use one `task_id` within a field for the same prediction objective, including studies with different inputs or models. For example, a constituent four-momentum model and a declustering-tree model both assign each jet a quenched or unquenched label. A model estimating the energy lost by a jet has another output and receives a separate task ID.

Use the following fields for each task–paper pair.

| field | description | note |
|---|---|---|
| `field` | Application area / Scientific field (e.g. pp, PbPb, GW, multimessenger) | Do not collapse pp/PbPb |
| `fm_scope` | Intended foundation-model input/context | E.g. particle-level jet constituents |
| `task_id` | Stable short identifier for the task | E.g. jet_classification_pp |
| `task_label` | Human-readable task name | |
| `task_type` | classification / regression / reconstruction / generation / other | |
| `observable_or_target` | Output predicted by the model in this study | E.g. quenched or unquenched label for each jet |
| `input_modality` | Data supplied to the model in this study | E.g. four-momenta of jet constituents
| `study_type` | experimental / phenomenological / methodological / unspecified | Where relevant; otherwise unspecified |
| `paper_id` | Paper identifier (arXiv ID, DOI, or INSPIRE key) | |
| `paper_url` | Direct link to paper | |
| `study_group_id` | Groups duplicate versions of the same study | |
| `match_rationale` | Why this study supports a potential benchmark for this FM scope | Required for every row |
| `evidence_text` | Quoted passage supporting the match | Required for supported decisions |
| `evidence_location` | Where in the paper (abstract / section / page) | |
| `decision` | candidate / supported / uncertain / excluded | |
| `review_status` | automated_proposal or human_reviewed | |
| `retrieval_method` | Retrieval method (e.g. lexical, LLM-assisted) and the query family that produced this row | For reproducibility |

### Your turn: one cell per paper
Fill one row per task–paper pair. For a supported row, cite the passage or figure reporting the model’s performance. 